## Обучение генеративной трансформерной модели с помощью `transformers`

В этой работе мы познакомимся на практике с процессом тренировки большой трансформерной языковой модели. Поскольку такая тренировка требует существенных вычислительных ресурсов, выполнять эту работу рекомендуется в Yandex DataSphere, в которой доступны вычислитльные узлы с одни или двумя графическими процессорами Tesla V100.

### Архитектура трансформеров

В рамках этой работы мы предполагаем, что вы уже знакомы с архитектурой трансформеров, например, по [статье из ML-хэндбука](https://academy.yandex.ru/handbook/ml/article/transformery). Также для первоначального знакомства рекомендую заметку [Jay Alammar. The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/), и её частичный [русскоязычный перевод](https://habr.com/ru/articles/486358/).

Мы не будем в рамках работы создавать архитетуру нейросети "с нуля". Если вам инетересно изучить реализацию трансформеров - рекомендую посмотреть на [NanoGPT](https://github.com/karpathy/nanoGPT). Подробно эта реализация разбирается в [этом видео](https://www.youtube.com/watch?v=kCc8FmEb1nY).

### Библиотека `transformers` и её друзья

Стандартом де факто в реализации трансформеров служит библиотека `transformers` от [HuggingFace](http://huggingface.co). Она содержит в себе реализацию большого количества используемых трансформерных архитектур, а также ряд полезных инструментов для их обучения. Многие инструменты также оформлены в виде отдельных библиотек, которые хорошо работают вместе:

* `tokenizers` - быстрая реализация различных токенизаторов, позволяющих разделять входной текст на токены
* `datasets` - манипулирование большими датасетами
* `evaluate` - вычисление различных метрик и оценка результатов обучения
* `accelerate` - реализация вычислений на множестве GPU и на вычислительных кластерах

Для начала, установим необходимые библиотеки:

In [6]:
%pip install --upgrade transformers==4.46.3 tokenizers==0.20.3 datasets==2.14.7 evaluate==0.4.1 accelerate==0.34.2 huggingface-hub==0.36.0 fsspec==2023.10.0 s3fs==2023.10.0 pyarrow==9.0.0

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 42.5 MB/s  0:00:00eta 0:00:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.7.0
    Uninstalling fsspec-2026.7.0:━━━━━━━━━━━━━━━ 0/4 [fsspec]
      Successfully uninstalled fsspec-2026.7.00m 0/4 [fsspec]
  Attempting uninstall: botocore━━━━━━━━━━━━━━━━ 0/4 [fsspec]
    Found existing installation: botocore 1.43.560/4 [fsspec]
    Uninstalling botocore-1.43.56:90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [botocore]
      Successfully uninstalled botocore-1.43.56━━━━━━━━━━━━━━━━━━━ 1/4 [botocore]
  Attempting uninstall: aiobotocore0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [botocore]
    Found existing installation: aiobotocore 3.9.0━━━━━━━━━━━━ 1/4 [botocore]
    Uninstalling aiobotocore-3.9.0:0m╺━━━━━━━━━━━━━━━━━━━ 2/4 [aiobotocore]
      Successfully uninstalled aiobotocore-3.9.0━━━━━━━━━━━━━━ 2/4 [aiobotocore]
  Attempting uninstall: s3

В текущем варианте при работе в DataSphere возникают проблемы при использовании файлового хранилища. Для решения проблем нам нужно установить последнюю версию библиотеки `s3fs`.

**ВНИМАНИЕ**: Данный ноутбук в полном режиме требует достаточно много вычислительных ресурсов (несколько часов обучения). Для более быстрого прогона установите в ячейке ниже значение переменной `RUN_MODE`:
* `smoke` - проверка синтаксиса, с минимальным обучением
* `demo` - демо-режим, прогоняется несколько эпох обучения
* `full` - полное обучение (но всё равно в демонстрационных рамках, но достаточно, чтобы увидеть эффект)

In [2]:
import importlib
import os
import site
import warnings

site.addsitedir(site.getusersitepackages())
importlib.invalidate_caches()

# Используем только PyTorch: так Transformers не загружает неиспользуемые TensorFlow/Flax-компоненты.
os.environ.pop("TRANSFORMERS_CACHE", None)
os.environ.setdefault("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Эти предупреждения приходят из сторонних библиотек и не относятся к операциям ноутбука.
warnings.filterwarnings(
    "ignore",
    message=r"The torchvision\.datapoints and torchvision\.transforms\.v2 namespaces are still Beta.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r"TypedStorage is deprecated.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r"Was asked to gather along dimension 0, but all input tensors were scalars;.*",
    category=UserWarning,
    module=r"torch\.nn\.parallel\._functions",
)

# Режимы запуска:
# smoke — 2 шага для проверки совместимости;
# demo — практический эксперимент за разумное время;
# full — длительное обучение для самостоятельного исследования.
RUN_MODE = "full"
SMOKE_TEST = RUN_MODE == "smoke"  # совместимость с пояснениями ниже
if RUN_MODE not in {"smoke", "demo", "full"}:
    raise ValueError("RUN_MODE должен быть smoke, demo или full")

RUN_SETTINGS = {
    "smoke": {
        "scratch": {"max_steps": 2},
        "finetune": {"max_steps": 2},
    },
    "demo": {
        "scratch": {"max_steps": 30},
        "finetune": {"max_steps": 100},
    },
    "full": {
        "scratch": {"num_train_epochs": 30},
        "finetune": {"num_train_epochs": 3},
    },
}
SCRATCH_TRAINING_LIMIT = RUN_SETTINGS[RUN_MODE]["scratch"]
FINETUNE_TRAINING_LIMIT = RUN_SETTINGS[RUN_MODE]["finetune"]
LOGGING_STEPS = 1 if RUN_MODE == "smoke" else 10

print(f"Режим: {RUN_MODE}")
print("Обучение с нуля:", SCRATCH_TRAINING_LIMIT)
print("Дообучение:", FINETUNE_TRAINING_LIMIT)


Режим: full
Обучение с нуля: {'num_train_epochs': 30}
Дообучение: {'num_train_epochs': 3}


### Подготовка датасета

В нашем примере, мы будем обучать виртуального Льва Толстого. Для этого, возьмём все основные романы писателя, и подготовим их них датасет. В качестве отправной точки будет использовать тексты из [библиотеки Мошкова](http://lib.ru). Соберем ссылки на романы Анна Каренина, Война и Мир и др. в один список:

In [9]:
urls = [
    "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0039.shtml",
    "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0040.shtml",
    "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0050.shtml",
    "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0060.shtml",
    "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0070.shtml",
    "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0080.shtml",
    "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0090.shtml",
    "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_1860_dekabristy.shtml",
]

Теперь скачаем все материалы и подготовим из них один большой текстовый файл. Для того нам понадобится убрать HTML-теги, а также несколько первоначальных строчек в каждом из файлов.

In [10]:
import html
import re

import requests


def download(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.text


# code borrowed from here: https://github.com/pallets/markupsafe/blob/0.23/markupsafe/__init__.py#L21
striptags_re = re.compile(r"(<!--.*?-->|<[^>]*>)")
metadata_tail_re = re.compile(r"\s+\d{5,}\s+\S+(?:\s+\d{4})?\s*$")
editorial_line_re = re.compile(
    r"^(?:Стр\.\s*\d+,\s*строка\s*\d+\.?|Вместо:|В Р\. В\.:)",
    flags=re.IGNORECASE,
)
editorial_suffix_re = re.compile(
    r"\s*Стр\.\s*\d+,\s*строка\s*\d+\..*$",
    flags=re.IGNORECASE,
)


def normalize_spaces(text):
    # HTML-сущность &nbsp; превращается в U+00A0; заменяем её до обучения.
    return text.replace("\u00a0", " ")


def show_text(text):
    # Показываем первый связный абзац без NBSP, повторных пробелов и служебных помет.
    text = editorial_suffix_re.sub("", normalize_spaces(text).strip())
    paragraph = re.split(r"\n\s*\n", text, maxsplit=1)[0]
    cleaned = " ".join(paragraph.split())
    print(metadata_tail_re.sub("", cleaned))


def to_text(s):
    return normalize_spaces(html.unescape(striptags_re.sub("", s)))


def beautify(s):
    lines = [x.strip() for x in s.split("\n") if x.strip()]
    for i in range(min(100, len(lines))):
        if lines[i] == "-->":
            break
    body = lines[i + 1 :] if i < 100 else lines
    body = [line for line in body if not editorial_line_re.match(line)]
    return "\n".join(body)


with open("dataset.txt", "w", encoding="utf-8") as f:
    for u in urls:
        text = beautify(to_text(download(u)))
        f.write(text + "\n\n")


В результате мы получили один большой файл `dataset.txt`, содержащий большой корпус текстов Льва Толстого.

### Хранение данных в Yandex DataSphere

При использовании Yandex DataSphere, у нас по умолчанию ограничен объем данных, которые мы можем хранить вместе с проектом. При необходимости, объем домашней директории проекта можно расширить, но хранение дополнительных данных в облаке становится платным. Поэтому при работе в команде особенно невыгодно хранить большие датасеты в домашних директориях всех проектов.

Обычно, большие объемы данных в облаке хранят в **объектном хранилище S3**. DataSphere позволяет легко подключаться к таким хранилищам, монтируя их как обычную директорию в проекте, после чего можно получить доступ к данным как к обычным файлам. Одно и то же хранилище может быть подключено к проектам разных пользователей, и они смогут пользоваться данными без их дублирования.

Однако, доступ в хранилище S3 не слишком быстрый, а для обучения сетей хочется отдавать данные как можно быстрее, не тормозя вычислительный процесс. Для этого в DataSphere предусмотрены **файловые хранилища** - это отдельные виртуальные накопители, которые можно легко подключать к различным вычислительным ресурсам.

Как файловые хранилища, так и хранилища S3 подключаются к проекту в виде отдельных директорий внутри `/home/jupyter`.

In [17]:
!ls /home/jupyter

datasets
datasphere
filestore
project
s3


Если хотите, вы можете в рамках нашего простого примера переместить файл `dataset.txt` в файловое хранилище, и поделиться им с другими участниками вашей организации (если вы работаете в группе).

### Токенизация

Нейросети работают с числами, поэтому первым этапом является токенизация текста, т.е. разбиение его на атомарные элементы, которые затем можно добавить в словарь, и представлять текст как последовательность индексов в словаре. Текст можно токенизировать по буквам, или по словам.

При построении современных генеративных сетей текст обычно разбивают на фрагменты таким образом, чтобы частота появления каждого фрагмента в тексте была примерно одинакова. Это лежит в основе т.н. Byte-Pair Encoding (BPE). Подробнее можно прочитать [в этой статье](https://huggingface.co/learn/nlp-course/chapter6/5?fw=pt).

Для обучения своего токенизатора используем библиотеку `tokenizers`:

In [4]:
import logging

import tokenizers as tok
import transformers as tr


class _MaxStepsMessageFilter(logging.Filter):
    def filter(self, record):
        return "max_steps is given, it will override any value given in num_train_epochs" not in record.getMessage()


logging.getLogger("transformers.trainer").addFilter(_MaxStepsMessageFilter())
tr.set_seed(42)

In [5]:
SPECIAL_TOKENS = ["[UNK]", "[PAD]", "[BOS]", "[EOS]"]

tokenizer = tok.Tokenizer(tok.models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = tok.pre_tokenizers.Whitespace()
trainer = tok.trainers.BpeTrainer(special_tokens=SPECIAL_TOKENS)
tokenizer.train(["dataset.txt"], trainer)
tokenizer.enable_padding(
    pad_id=tokenizer.token_to_id("[PAD]"),
    pad_token="[PAD]",
)

А данном случае мы используем два специальных токена - `[UNK]` для представления неизвестного токена (такое случится, если на вход попадёт символ, который токенизатор не видел при обучении), и `[PAD]` для **паддинга** - он используется, если нужно дополнить последовательность до определённой длины.

Вот как можно закодировать входной текст:

In [6]:
tokenizer.encode("Иван Сигизмундович подошел к окну и закашлялся. Вечерело.").tokens

['Иван',
 'С',
 'иг',
 'изму',
 'н',
 'до',
 'вич',
 'подошел',
 'к',
 'окну',
 'и',
 'за',
 'кашлялся',
 '.',
 'Вечер',
 'ело',
 '.']

Видим, что популярные слова токенизируются целиком, а те, которые встречаются в тексте редко или не встречаются вовсе - разбиваются на фрагменты.

### Генеративные трансформеры

Для генерации текста используются архитектуры GPT - Generative Pre-trained Transformers. В то время как полноценные трансформеры являются энкодер-декодерной архитектурой, т.е. могут решать задачи преобразования одного вида последовательности в другую, GPT является только декодером, т.к. способно прогнозировать распределение вероятности следующего слова по начальной части последовательности.

Мы используем архитектуру GPT-2, которая, с одной стороны, не слишком огромна, а с другой - может неплохо обучиться. Сперва попробуем натренировать такую архитетуру "с нуля".

Дла начала нам потребуется преобразовать наш токенизатор к объекту `ttokenizer`, который понимает библиотека transformers.

In [7]:
vocab = tokenizer.get_vocab()
ttokenizer = tr.PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[BOS]",
    eos_token="[EOS]",
)
len(vocab)

30000

Теперь создадим непосредственно нейросетевую модель GPT2. При этом основные параметры (количество слоёв, количество голов внимания и т.д. оставим по умолчанию.

In [21]:
config = tr.GPT2Config(
    vocab_size=len(vocab),
    bos_token_id=ttokenizer.bos_token_id,
    eos_token_id=ttokenizer.eos_token_id,
    pad_token_id=ttokenizer.pad_token_id,
)
gpt = tr.GPT2LMHeadModel(config)

Веса вновь созданной модели инициализируются случайным образом, поэтому если мы попросим такую модель сгенерировать текст - получится бессмыслица:

In [22]:
res = gpt.generate(
    **ttokenizer("Мне нравится ", return_tensors="pt"),
    max_new_tokens=50,
    top_k=3,
    do_sample=True,
)
print("Случайно инициализированная модель — бессмысленный текст здесь ожидаем:")
show_text(
    ttokenizer.decode(
        res[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )
)

Случайно инициализированная модель — бессмысленный текст здесь ожидаем:
Мне нравится каторжных зову понятиях понятиях чиби toi кофе отряд Андреевичу отказ позволю важности высоты позволю позволю подурнела подурнела подурнела soldat представлением представлением представлением женным женным женным женным надзира надзира il представлением позволю позволю позволю позволю революцио зату поразительно поразительно ручку хую светские Денисову soldat soldat решалась решалась холостя вскри Денисову Денисову


Теперь нам надо научиться подавать на вход модели фрагменты текста для обучения. Для этого существует библиотека `datasets`, входящее в семейство трансформерных библиотек HuggingFace. Помимо того, что эта библиотека умеет работать с разными форматами входных датасетов, она также интегрирована с HuggingFace Hub, и может в одну строчку загружать множество имеющихся на этом сайте датасетов.

В нашем случае мы загрузим датасет из текстового файла:

In [5]:
import datasets

dataset = datasets.load_dataset("text", data_files="dataset.txt")
dataset["train"][13]

{'text': 'Он взял своею большою рукой меня за руку, и пожал так крепко, честно, только что не больно. Я думала, что он поцелует мою руку, и нагнулась было к нему, но он еще раз пожал мне руку и прямо в глаза посмотрел своим твердым и веселым взглядом.'}

Далее нам необходимо научиться токенизировать датасет, т.е. преобразовывать в числовые тензоры, которые затем мы будем подавать на вход нейросети в процессе обучения. Для этого опишем фукнцию `tokenize`, которая будет возвращать словарь с несколькими полями:

* `input_ids` - это собственно номера слов входной последовательности в словаре
* `token_type_ids` - содержит нули. Это поле используется в более сложных сценариях, например, когда мы тренируем сеть отвечать на вопросы по тексту. В этом случае нам нужно подать на вход текст + вопрос, и это поле позволяет различать между несколькими разными по смыслу фрагментами входной последовательности
* `atttention_mask` показывает, какая часть входной последовательности значима. Для организации последовательности в minibatch нам может потребоваться дополнить последовательность до максимальной длины, и поле `attention_mask` содержит 1 в тех позициях, которые соответствуют исходной последовательности

Такой формат входных данных типичен для трансформерной архитектуры. Также мы передаем последовательность значений целевой переменной `labels`, но поскольку наша задача - это генерация текста, то в качестве `labels` мы передаём копию исходного текста. 

In [11]:
def tokenize(x):
    x = ttokenizer(x["text"])
    x["labels"] = x["input_ids"].copy()
    return x


ds = dataset.map(tokenize, batched=True, remove_columns=["text"])
ds["train"][0]

Map: 100%|██████████| 28453/28453 [00:04<00:00, 6219.39 examples/s]


{'input_ids': [10083, 8328, 2773],
 'token_type_ids': [0, 0, 0],
 'attention_mask': [1, 1, 1],
 'labels': [10083, 8328, 2773]}

Для обучения лучше всего использовать длинные фрагменты текста, поэтому мы сгруппируем все последовательности токенов в блоки размером `block_size`. Для этого мы сначала сконкатенируем все последовательности, а потом разобъем их на блоки. В данном случае мы не будем даже разбивать последовательность на слова и/или предложения - как показывает практика, такой упрощенный подход также даёт хорошие результаты.


In [13]:
from itertools import chain

block_size = 1024

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

dsb = ds.map(group_texts, batched=True)

Map: 100%|██████████| 2846/2846 [00:00<00:00, 8780.14 examples/s]


Теперь мы готовы к обучению! Для задания параметров обучения мы создаём объект `TrainingArguments`, в котором задаем директорию, куда будут записываться промежуточные результаты обучения, число эпох, скорость обучения и т.д. Затем на основе этих параметров создаём объект `Trainer`.

Обратите внимание, что размер записываемой на диск сети GPT-2 может быть весьма большим (около 1.4 Gb), что может привести к исчерпанию размера вашей домашней директории в DataSphere. Исходя из этого лучше выбирать параметры `save_steps` и `num_train_epochs` таким образом, чтобы количество записываемых на диск чекпоинтов не превышало 3-5 шт.

Для начала стоит попробовать пообучать сеть в течение 30-90 минут, чтобы увидеть, что она начинает складывать слова более менее правдоподобно.

В ноутбуке есть три режима. 'smoke' выполняет по два шага и проверяет только техническую исправность. Режим 'demo' используется по умолчанию: он ограничивает обучение с нуля 30 шагами, а дообучение — 100 шагами. Режим 'full' оставлен для длительного самостоятельного эксперимента. Результат дообучения ниже оценивается на отложенной части корпуса, поэтому вывод не зависит от одной случайно сгенерированной фразы.


In [32]:
targs = tr.TrainingArguments(
    output_dir="gpt2-scratch",
    **SCRATCH_TRAINING_LIMIT,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    save_strategy="no",
    logging_steps=LOGGING_STEPS,
    fp16=True,
    report_to="none",
)
trainer = tr.Trainer(
    gpt,
    args=targs,
    train_dataset=dsb["train"],
    processing_class=ttokenizer,
    data_collator=tr.default_data_collator,
)

In [ ]:
trainer.train()

  0%|          | 10/5070 [00:10<1:25:16,  1.01s/it]

{'loss': 7.937, 'grad_norm': 1.5700181722640991, 'learning_rate': 9.861932938856016e-07, 'epoch': 0.06}


  0%|          | 20/5070 [00:20<1:25:03,  1.01s/it]

{'loss': 7.9486, 'grad_norm': 1.3731402158737183, 'learning_rate': 1.9723865877712033e-06, 'epoch': 0.12}


  1%|          | 30/5070 [00:30<1:25:01,  1.01s/it]

...

100%|█████████▉| 5060/5070 [1:25:17<00:10,  1.01s/it]

{'loss': 4.2868, 'grad_norm': 4.2155609130859375, 'learning_rate': 1.0957703265395574e-07, 'epoch': 29.94}


100%|██████████| 5070/5070 [1:25:27<00:00,  1.01s/it]

{'loss': 4.2719, 'grad_norm': 5.950577735900879, 'learning_rate': 0.0, 'epoch': 30.0}
{'train_runtime': 5127.2729, 'train_samples_per_second': 7.887, 'train_steps_per_second': 0.989, 'train_loss': 5.292667208338631, 'epoch': 30.0}


TrainOutput(global_step=5070, training_loss=5.292667208338631, metrics={'train_runtime': 5127.2729, 'train_samples_per_second': 7.887, 'train_steps_per_second': 0.989, 'total_flos': 2.1170925600768e+16, 'train_loss': 5.292667208338631, 'epoch': 30.0})

Теперь посмотрим, как работает генерация:

In [35]:
res = gpt.generate(
    **ttokenizer("Пьер закашлялся и", return_tensors="pt").to("cuda"),
    max_new_tokens=150,
    do_sample=True,
    repetition_penalty=1.1,
)

show_text(
    ttokenizer.decode(
        res[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )
)

Пьер за кашлялся и в свою сторону Москвы и,. Пьер не был не думал от него времени его во всем, этот период своей цели, по его ; но Пьер был тогда как будто теперь казалось, как бы только не может быть ни того, что она теперь это его или не было. Но в Москве, у нас не было то же утро -- Я вас, а я чего! -- сказал он : " да и говорил Пьер. Он чувствовал, как бы то, как говорят, чтоб иметь это был всегда не мог, он себе, чтò такое утро! Он еще для других людей со всеми его и чувствовал, чтò можно сделать не он будет. Так, всё только то, что он сделал всё - таки хотела сказать, но все люди этого сделать это. -- Не


Кажется, что сгенерированный текст пока ещё не слишком осмысленный. Но сравните его с первоначальным текстом, сгенерированным необученной нейросетью - в нём почти не было корректных грамматических конструкций. За примерно час обучения сеть уже стала неплохо понимать, какие слова хорошо сочетаются друг с другом, и в целом начала говорить более осмысленно. Помните, что трансформерная модель - сложная, и для обучения полноценной GPT-2 "с нуля" требуются сотни и тысячи GPU-часов.

> Прежде, чем переходить к следующим экспериментам, очистим память. Если вдруг на следующем этапе возникнет переполнение памяти GPU, может потребоваться перезапуск ядра ноутбука - выберите к меню Kernel -> Restart Kernel

In [36]:
import gc
import torch

gpt = None
trainer = None
gc.collect()
torch.cuda.empty_cache()

## До-обучение GPT-2

За приемлемое время сложно достичь приемлемого качества обучения трансформера, поэтому обычно используют предобученные модели (поэтому в названии GPT и фигурирует слово *Pretrained*), которые уже научились "читать" на нужном языке, и их необходимо лишь немного "доучить" под требуемую предметную область или стиль. В этом случае процесс обучения модели почти не отличается от того, что мы делали ранее - с той лишь разницей, что необходимо использовать токенизатор, который использовался при обучении исходной модели.

Для начала, загрузим предобученную модель **ruGPT** и соответствующий токенизатор, и посмотрим, как эта модель умеет продолжать текст:

In [11]:
tokenizer = tr.AutoTokenizer.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")
tokenizer.pad_token = tokenizer.eos_token
gpt = tr.GPT2LMHeadModel.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")
gpt.config.pad_token_id = tokenizer.pad_token_id
gpt.generation_config.pad_token_id = tokenizer.pad_token_id

PROMPTS = [
    "Пьер вошёл в комнату и",
    "Князь Андрей посмотрел на",
    "Анна сказала, что",
]

def generate_examples(model):
    model.eval()
    device = next(model.parameters()).device
    examples = {}
    for prompt in PROMPTS:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            result = model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                repetition_penalty=1.2,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.pad_token_id,
            )
        examples[prompt] = normalize_spaces(
            tokenizer.decode(
                result[0],
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )
        )
    return examples

before_examples = generate_examples(gpt)
print("Контрольные генерации до дообучения:")
for prompt, text in before_examples.items():
    print(f"\nПромпт: {prompt}")
    show_text(text)


Контрольные генерации до дообучения:

Промпт: Пьер вошёл в комнату и
Пьер вошёл в комнату и увидел, что его жена сидит на кровати.

Промпт: Князь Андрей посмотрел на
Князь Андрей посмотрел на него с удивлением. — Ты, кажется, не знаешь?

Промпт: Анна сказала, что
Анна сказала, что у нее есть дочь.


На самом деле качество модели *очень сильно* зависит от количества параметров, и тот факт, что мы взяли модель **ruGPTsmall** сказывается на качестве текста. Но зато и процесс обучения будет существенно быстрее!

Поскольку мы теперь используем другой токенизатор, то нам нужно заново токенизировать датасет:

Чтобы сравнение было проверяемым, разделим корпус на обучающую и отложенную части. Значение 'eval_loss' и рассчитанная из него perplexity будут измерены для одной и той же предобученной модели до и после дообучения. Генерации по нескольким фиксированным промптам останутся дополнительной качественной иллюстрацией.


In [14]:
raw_dataset = datasets.load_dataset(
    "text",
    data_files="dataset.txt",
    split="train",
)
dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)

ds = dataset.map(
    lambda batch: tokenizer(batch["text"]),
    batched=True,
    remove_columns=["text"],
)
dsb = ds.map(group_texts, batched=True)

print("Обучающих блоков:", len(dsb["train"]))
print("Проверочных блоков:", len(dsb["test"]))


Обучающих блоков: 1333
Проверочных блоков: 151


Сам по себе процесс запуска обучения и указания параметров ничем не отличается от обучения трансформерной модели "с нуля". Возможно, при до-обучении имеет смысл указывать чуть более низкий `learning_rate`.

In [15]:
import math

targs = tr.TrainingArguments(
    output_dir="gpt2-finetune",
    **FINETUNE_TRAINING_LIMIT,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    save_strategy="no",
    logging_steps=LOGGING_STEPS,
    per_device_eval_batch_size=8,
    fp16=True,
    report_to="none",
)
trainer = tr.Trainer(
    model=gpt,
    args=targs,
    train_dataset=dsb["train"],
    eval_dataset=dsb["test"],
    processing_class=tokenizer,
    data_collator=tr.default_data_collator,
)

baseline_metrics = trainer.evaluate()
baseline_loss = baseline_metrics["eval_loss"]
baseline_perplexity = math.exp(baseline_loss)

train_result = trainer.train()

final_metrics = trainer.evaluate()
final_loss = final_metrics["eval_loss"]
final_perplexity = math.exp(final_loss)

print("\nСравнение на отложенной выборке")
print(f"eval_loss:  {baseline_loss:.4f} -> {final_loss:.4f}")
print(f"perplexity: {baseline_perplexity:.2f} -> {final_perplexity:.2f}")
print(f"Снижение eval_loss: {baseline_loss - final_loss:.4f}")

if RUN_MODE != "smoke" and final_loss >= baseline_loss:
    raise AssertionError(
        "На отложенной выборке качество не улучшилось. "
        "Повторите запуск или увеличьте бюджет дообучения."
    )


  2%|▏         | 10/501 [00:04<03:36,  2.26it/s]

{'loss': 3.8719, 'grad_norm': 1.3278892040252686, 'learning_rate': 9.803921568627451e-06, 'epoch': 0.06}


  4%|▍         | 20/501 [00:09<03:27,  2.32it/s]

{'loss': 3.7901, 'grad_norm': 1.2855746746063232, 'learning_rate': 1.9607843137254903e-05, 'epoch': 0.12}


  6%|▌         | 30/501 [00:13<03:21,  2.33it/s]

...

{'loss': 3.2621, 'grad_norm': 1.091622233390808, 'learning_rate': 1.1111111111111112e-07, 'epoch': 2.99}


100%|██████████| 501/501 [03:34<00:00,  2.34it/s]


{'train_runtime': 214.502, 'train_samples_per_second': 18.643, 'train_steps_per_second': 2.336, 'train_loss': 3.404230616049852, 'epoch': 3.0}


100%|██████████| 19/19 [00:03<00:00,  6.30it/s]


Сравнение на отложенной выборке
eval_loss:  3.7499 -> 3.3234
perplexity: 42.52 -> 27.75
Снижение eval_loss: 0.4265


Теперь повторим те же детерминированные генерации. Это качественная иллюстрация: отдельные продолжения могут совпасть, даже если модель обучилась. Основное свидетельство эффекта — изменение 'eval_loss' и perplexity на отложенной выборке.


In [16]:
after_examples = generate_examples(gpt)

print("Контрольные генерации после дообучения:")
for prompt in PROMPTS:
    print(f"\nПромпт: {prompt}")
    print("До:   ", end="")
    show_text(before_examples[prompt])
    print("После:", end=" ")
    show_text(after_examples[prompt])
    print("Текст изменился:", before_examples[prompt] != after_examples[prompt])


Контрольные генерации после дообучения:

Промпт: Пьер вошёл в комнату и
До:   Пьер вошёл в комнату и увидел, что его жена сидит на кровати.
После: Пьер вошёл в комнату и, не раздеваясь, сел на диван.-- Я думаю, что это очень хорошо, -- сказал он с улыбкой. -- Но я боюсь, что вы будете недовольны этим. Вы знаете, как мне
Текст изменился: True

Промпт: Князь Андрей посмотрел на
До:   Князь Андрей посмотрел на него с удивлением. — Ты, кажется, не знаешь?
После: Князь Андрей посмотрел на него.-- Я не могу, -- сказал он и опять стал ходить по комнате. Он чувствовал себя виноватым за то, что так долго молчал; но ему было жалко его.Он встал с дивана
Текст изменился: True

Промпт: Анна сказала, что
До:   Анна сказала, что у нее есть дочь.
После: Анна сказала, что она не может быть счастлива.-- Я знаю, -- сказал он и опять стал ходить по комнате. -- Но я боюсь за тебя... Ты знаешь, как это бывает? Когда ты чувствуешь себя виноватым
Текст изменился: True


Если 'eval_loss' и perplexity снизились, модель стала лучше предсказывать продолжение текста на данных, которых не видела при обучении. Это корректнее, чем вывод по одной красивой фразе. В режиме 'smoke' изменение метрик может быть минимальным: этот режим предназначен только для проверки исполнения. В режимах 'demo' и 'full' мы ожидаем измеримый эффект, но не требуем, чтобы изменился каждый детерминированный пример.

## Параллелизация обучения

Надеюсь, вы убедились, что на DataSphere можно обучать достаточно мощные модели, однако время, затрачиваемое на обучение, всё ещё остаётся большим. Чтобы ускорить этот процесс, обычно используют параллельное обучение на нескольких GPU одновременно.

Самым распространённым вариантом параллелизма является параллелизм по данным (Data Parallel Training), в котором на каждый из обучающих GPU подаётся свой поток данных (т.е. своя часть исходного датасета). При этом на каждом обучающем шаге каждый GPU вычисляет свой градиент ошибки, которые затем усредняются и используются для синхронного обновления моделей на всех обучающих процессорах.

Различают два варианта обучения на нескольких GPU:
* **Data Parallel** - обычно используется, когда несколько GPU установлены на одном компьютере. В этом случае используется почти такой же код обучения на Python, как для однопроцессорного варианта, модель оборачивается в класс `torch.nn.DataParallel`, и минибатч распределяется по нескольким доступным на данном компьютере GPU.
* **Distributed Data Parallel** используется в более общем случае, когда есть кластер из компьютеров с GPU.

Подробнее про параллельное обучение можно прочитать [в руководстве PyTorch](https://pytorch.org/docs/stable/distributed.html#distributed-basics).


## Заключение

Одна из целей данной работы заключалась в том, чтобы продемонстрировать, что обучение сложных языковых моделей с помощью современных библиотек является сравнительно простой задачей - но требующей значительных вычислительных ресурсов. Как только мы выходим за рамки вычислений, которые можно сделать за несколько часов на общедоступных инструментах типа Google Colab - у нас возникает потребность в облачных вычислительных ресурсах.

Yandex DataSphere обеспечивает легкий переход от локального Jupyter Notebook или публичного облака Google Colab / Kaggle к выделенной облачной инфраструктуре в Yandex Cloud. В DataSphere вы можете:

* легко настроить подключения к облачным хранилищам данных, 
* взаимодействовать с другими участниками проекта
* использовать GitHub для контроля версий кода
* бережливо расходовать ресурсы благодаря режиму Serverless или возможности легкого переключения между виртуальными вычислителями

Для эффективной работы в DataSphere в ней необходимо немного привыкнуть, но когда этап привыкания пройдёт - вы сможете эффективно пользоваться этим инструментом и получать удовольствие от работы в нём!